In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
import pandas as pd

df = pd.read_csv(r'C:\Users\SSC\Desktop\job_intel\data\processed\jobs_clean.csv')

print(df.columns.tolist())

['title', 'jobId', 'currency', 'jobUploaded', 'companyName', 'tagsAndSkills', 'experience', 'salary', 'location', 'companyId', 'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience']


In [3]:
# fill missing tags with empty strings
df['tagsAndSkills'] = df['tagsAndSkills'].fillna('')

# Initialize tfidfvectorizer for skills matching

tfidf = TfidfVectorizer(stop_words='english')
job_skills_matrix = tfidf.fit_transform(df['tagsAndSkills'])


In [6]:
def match_resume_and_find_gaps(user_skills_string, top_n=5):
    """
    Matches user skills using Cosine Similarity and performs 
    Phrase-level Skill-Gap Analysis (ignoring single-word noise and titles).
    """
    user_vec = tfidf.transform([user_skills_string])
    similarity_scores = cosine_similarity(user_vec, job_skills_matrix).flatten()
    top_indices = similarity_scores.argsort()[::-1][:top_n]
    
    results = []
    user_text_lower = user_skills_string.lower()
    
    # Noise words ya generic titles jo skills nahi hain
    noise_words = {'data', 'machine', 'science', 'scientist', 'developer', 'engineer', 'analyst'}
    
    for idx in top_indices:
        job_row = df.iloc[idx]
        job_title = job_row.get('title', 'N/A')
        job_company = job_row.get('companyName', 'N/A')
        job_loc = job_row.get('location', 'N/A')
        job_exp = job_row.get('experience', 'N/A')
        
        job_skills_raw = str(job_row['tagsAndSkills'])
        
        # 1. Comma se split karke poore phrases lo
        job_skill_phrases = [s.strip().lower() for s in job_skills_raw.split(',') if s.strip()]
        
        missing_skills = []
        for phrase in job_skill_phrases:
            # 2. Check if the full phrase is NOT in user resume text
            if phrase not in user_text_lower:
                # 3. Filter out standalone noise words or single-word fragments
                if phrase not in noise_words and len(phrase) > 2:
                    missing_skills.append(phrase)
                    
        match_percentage = round(similarity_scores[idx] * 100, 2)
        
        results.append({
            'Job Title': job_title,
            'Company': job_company,
            'Location': job_loc,
            'Experience': job_exp,
            'Match Score (%)': match_percentage,
            'Missing Skills': ', '.join(list(dict.fromkeys(missing_skills))[:5]) # Unique missing skills, top 5
        })
        
    return pd.DataFrame(results)

# Test again with sample skills
sample_user_skills = "Python, SQL, Machine Learning, Pandas, Scikit-learn"

print(f"--- Updated Top Matching Jobs & Clean Skill-Gaps ---\n")
recommendations = match_resume_and_find_gaps(sample_user_skills, top_n=5)
display(recommendations)

--- Updated Top Matching Jobs & Clean Skill-Gaps ---



,Job Title,Company,Location,Experience,Match Score (%),Missing Skills
0,Artificial Intelligence and Machine Learning G...,Aveva,"Hyderabad, Bengaluru",3-7 Yrs,74.67,"numpy, tensorflow, aiml"
1,Data Scientist,NKT Operations,Chennai,5-10 Yrs,73.52,"numpy, data science, azure, statistics"
2,Data Scientist,Gokloud Software Solutions,Remote,3-7 Yrs,69.15,"deep learning, tensorflow"
3,Data Scientist,Tsit Digital Technologies,"Hybrid - Hyderabad, Chennai, Bengaluru",5-10 Yrs,66.07,data scientist
4,Lead Data Scientist,Mastercard,Pune,9-12 Yrs,65.72,"data science, software engineering, data analy..."


In [7]:
import os
import joblib

# 1. Ensure models directory exists in root
models_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'models'))
os.makedirs(models_dir, exist_ok=True)

# 2. Package the vectorizer, matrix, and essential dataframe columns
matcher_data = {
    'tfidf': tfidf,
    'job_skills_matrix': job_skills_matrix,
    'df': df[['title', 'companyName', 'location', 'experience', 'tagsAndSkills']]
}

# 3. Save to models folder
matcher_path = os.path.join(models_dir, 'resume_matcher_pipeline.pkl')
joblib.dump(matcher_data, matcher_path)

print(f"Success! Resume Matcher pipeline saved at: {matcher_path}")

Success! Resume Matcher pipeline saved at: c:\Users\SSC\Desktop\job_intel\models\resume_matcher_pipeline.pkl
